# Dynamic Qwen Model Training with GRPO

This notebook implements the same training process as the dynamic_qwen0.py script, allowing for interactive execution and visualization of the training process.

In [ ]:
import os                                                                            
os.environ["CUDA_VISIBLE_DEVICES"] = "6"                                                    
print(f"Using GPU: {os.environ['CUDA_VISIBLE_DEVICES']}")  

In [ ]:
import os
import wandb
import logging
import json
from datasets import load_dataset, concatenate_datasets, Dataset, load_from_disk
from datetime import datetime
from unsloth import is_bfloat16_supported
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)
import sys
from trl import GRPOConfig, GRPOTrainer
from transformers import TrainerCallback
import re
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, HTML

# Ensure the project root is in sys.path for imports
project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
    
from grpo.config import RewardConfig
from grpo.dynamic_reward import DynamicReward
from utils.similarity_checker import SolutionSimilarityChecker
from utils.data_preparation import prepare_combined_data
from utils.agents import (
    FULLSOLUTION_SYSTEM_PROMPT, 
    COMPLETION_SYSTEM_PROMPT,
    PROGRAMMER_SYSTEM_PROMPT
)

## Setup Logging

First, let's set up logging to track our progress.

In [ ]:
def setup_logging(model_type: str) -> logging.Logger:
    """Setup logging configuration"""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_dir = f"logs/{model_type}"
    os.makedirs(log_dir, exist_ok=True)
    
    logger = logging.getLogger('dynamic_grpo')
    
    # Clear any existing handlers to prevent duplicate logging
    if logger.handlers:
        logger.handlers.clear()
        
    logger.setLevel(logging.INFO)
    
    file_handler = logging.FileHandler(
        f"{log_dir}/notebook_training_{timestamp}.log"
    )
    file_handler.setFormatter(logging.Formatter(
        '%(asctime)s - %(message)s',
        datefmt='%Y-%m-%d %H:%M:%S'
    ))
    logger.addHandler(file_handler)
    logger.addHandler(logging.StreamHandler())
    return logger

# Initialize logger
model_type = "dynamic_0"
logger = setup_logging(model_type)
logger.info("Notebook started")

## Define Logging Callback

Let's define a callback to track training progress and visualize metrics.

In [ ]:
class LoggingCallback(TrainerCallback):
    """Callback for logging training metrics"""
    def __init__(self, reward_func, logger, save_frequency=100):
        self.reward_func = reward_func
        self.save_frequency = save_frequency
        self.step = 0
        self.logger = logger
        self.metrics_history = {
            'step': [],
            'reward': [],
            'average_reward': [],
            'solution_reward_uses': [],
            'completion_reward_uses': [],
            'programming_reward_uses': []
        }
        
    def on_log(self, args, state, control, logs=None, **kwargs):
        self.step += 1
        
        if logs and 'rewards/0' in logs and hasattr(self.reward_func, 'stats'):
            # Print detailed stats to console/log file
            self.logger.info("\n" + "="*50)
            self.logger.info(f"Step {self.step} - Reward Stats Summary:")
            
            # Get and log the stats summary
            stats_summary = self.reward_func.stats.get_summary()
            self.logger.info(stats_summary)
            self.logger.info("="*50 + "\n")
            
            # Track metrics for visualization
            self.metrics_history['step'].append(self.step)
            self.metrics_history['reward'].append(logs.get('rewards/0', 0))
            self.metrics_history['average_reward'].append(
                self.reward_func.stats.reward_components.get('average_reward', 0.0))
            
            # Track reward type usage
            self.metrics_history['solution_reward_uses'].append(
                self.reward_func.stats.reward_components.get('solution_reward_uses', 0))
            self.metrics_history['completion_reward_uses'].append(
                self.reward_func.stats.reward_components.get('completion_reward_uses', 0))
            self.metrics_history['programming_reward_uses'].append(
                self.reward_func.stats.reward_components.get('programming_reward_uses', 0))
            
            # Key performance metrics for wandb
            wandb_stats = {
                'reward': logs['rewards/0'],
                'average_reward': self.reward_func.stats.reward_components.get('average_reward', 0.0),
                'total_batches': self.reward_func.stats.total_batches,
                'total_examples': self.reward_func.stats.total_examples
            }
            
            # Add dynamic reward specific metrics
            if 'solution_reward_uses' in self.reward_func.stats.reward_components:
                wandb_stats['solution_reward_uses'] = self.reward_func.stats.reward_components['solution_reward_uses']
            if 'completion_reward_uses' in self.reward_func.stats.reward_components:
                wandb_stats['completion_reward_uses'] = self.reward_func.stats.reward_components['completion_reward_uses']
            if 'programming_reward_uses' in self.reward_func.stats.reward_components:
                wandb_stats['programming_reward_uses'] = self.reward_func.stats.reward_components['programming_reward_uses']
                
            # Track example types in the batch
            if hasattr(state, 'train_dataloader') and state.train_dataloader is not None:
                try:
                    # Get current batch
                    batch_idx = (state.global_step - 1) % len(state.train_dataloader)
                    current_batch = list(state.train_dataloader)[batch_idx]
                    
                    # Count example types if available
                    if 'example_type' in current_batch:
                        example_types = current_batch['example_type']
                        solution_count = sum(1 for t in example_types if t == 'solution')
                        completion_count = sum(1 for t in example_types if t == 'completion')
                        wait_count = sum(1 for t in example_types if t == 'wait')
                        programming_count = sum(1 for t in example_types if t == 'programming')
                        
                        wandb_stats['solution_examples'] = solution_count
                        wandb_stats['completion_examples'] = completion_count
                        wandb_stats['wait_examples'] = wait_count
                        wandb_stats['programming_examples'] = programming_count
                except Exception as e:
                    self.logger.warning(f"Could not track example types: {str(e)}")
            
            # Add all stats from reward_components to wandb
            for key, value in self.reward_func.stats.reward_components.items():
                wandb_stats[f'reward_components/{key}'] = value
                
            # Add group stats
            for key, value in self.reward_func.stats.group_stats.items():
                wandb_stats[f'group_stats/{key}'] = value
                
            # Add step stats
            for key, value in self.reward_func.stats.step_stats.items():
                wandb_stats[f'step_stats/{key}'] = value
                
            # Add similarity stats
            for key, value in self.reward_func.stats.similarity_stats.items():
                wandb_stats[f'similarity_stats/{key}'] = value
                
            # Add programming stats
            for key, value in self.reward_func.stats.programming_stats.items():
                wandb_stats[f'programming_stats/{key}'] = value
                
            # Add reward distribution
            if hasattr(self.reward_func.stats, 'reward_distribution') and self.reward_func.stats.reward_distribution:
                # Only log the top 10 most common rewards to avoid cluttering wandb
                sorted_rewards = sorted(
                    self.reward_func.stats.reward_distribution.items(), 
                    key=lambda x: self.reward_func.stats.reward_distribution[x[0]], 
                    reverse=True
                )[:10]
                
                for reward, count in sorted_rewards:
                    wandb_stats[f'reward_distribution/{reward}'] = count
            
            # Update logs with our metrics
            logs.update(wandb_stats)
            
    def plot_metrics(self):
        """Plot the tracked metrics"""
        if not self.metrics_history['step']:
            print("No metrics to plot yet.")
            return
            
        # Create a figure with multiple subplots
        fig, axs = plt.subplots(2, 1, figsize=(12, 10))
        
        # Plot rewards
        axs[0].plot(self.metrics_history['step'], self.metrics_history['reward'], label='Batch Reward')
        axs[0].plot(self.metrics_history['step'], self.metrics_history['average_reward'], label='Average Reward')
        axs[0].set_title('Rewards Over Training')
        axs[0].set_xlabel('Step')
        axs[0].set_ylabel('Reward Value')
        axs[0].legend()
        axs[0].grid(True, alpha=0.3)
        
        # Plot reward type usage
        axs[1].plot(self.metrics_history['step'], self.metrics_history['solution_reward_uses'], label='Solution Reward')
        axs[1].plot(self.metrics_history['step'], self.metrics_history['completion_reward_uses'], label='Completion Reward')
        axs[1].plot(self.metrics_history['step'], self.metrics_history['programming_reward_uses'], label='Programming Reward')
        axs[1].set_title('Reward Type Usage')
        axs[1].set_xlabel('Step')
        axs[1].set_ylabel('Count')
        axs[1].legend()
        axs[1].grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()

## Main Training Setup

Now let's set up the main training configuration and components.

In [ ]:
# Configuration
model_name = "/Home/stat/laschos/math/AIMO2_initial/models/wait_2/20250304_194943"
dataset_name = "Metaskepsis/completion"

# Initialize config
reward_config = RewardConfig(model_type=model_type)
reward_config.group_diversity_bonus = 2  # Increased from 1.0

# Setup
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_dir = f"train_results/{reward_config.model_type}/{timestamp}"
wandbname = f"{model_type}, DB={reward_config.group_diversity_bonus}, {model_name}, {dataset_name}, {timestamp}"

# Initialize wandb
use_wandb = False  # Set to True if you want to use wandb
if use_wandb:
    wandb.init(
        project="grpo",
        name=wandbname,
        config={
            "model_type": reward_config.model_type,
            "dataset": dataset_name,
            "base_reward": 3.0,
            "diversity_bonus": 0.3,
            "step_continuity_reward": 0.5
        }
    )
    report_to = "wandb"
else:
    report_to = "none"

# Initialize similarity checker first
similarity_checker = SolutionSimilarityChecker(reward_config)

# Initialize dynamic reward function
reward_func = DynamicReward(reward_config, similarity_checker)
logger.info("\nInitialized DynamicReward:")
logger.info(f"Has stats object: {hasattr(reward_func, 'stats')}")

# Print initial stats configuration
if hasattr(reward_func, 'stats'):
    logger.info("Initial stats configuration:")
    for category in ['reward_components', 'group_stats', 'step_stats', 'similarity_stats']:
        if hasattr(reward_func.stats, category):
            stats_dict = getattr(reward_func.stats, category)
            logger.info(f"{category}: {stats_dict}")
else:
    logger.warning("No stats object found in reward_func!")

## Load and Initialize Model

Now let's load the Qwen model using Unsloth for faster training.

In [ ]:
# Load model
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=4596,
    fast_inference=True,
    load_in_4bit=False,
    use_gradient_checkpointing="unsloth",
    gpu_memory_utilization=0.6,
    max_lora_rank=64)
    
# Function to count tokens in a string
def count_tokens(text):
    return len(tokenizer.encode(text))
    
# Calculate token counts for system prompts
solver_prompt_tokens = count_tokens(FULLSOLUTION_SYSTEM_PROMPT)
completion_prompt_tokens = count_tokens(COMPLETION_SYSTEM_PROMPT)
logger.info(f"Solver system prompt: {solver_prompt_tokens} tokens")
logger.info(f"Completion system prompt: {completion_prompt_tokens} tokens")

# Configure LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                   "gate_proj", "up_proj", "down_proj"],
    lora_alpha=64,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None
)

## Load and Prepare Dataset

Let's load the dataset and prepare it for training.

In [ ]:
def get_questions(split="train") -> Dataset:
    """Load and format dataset with full solution, completion, programming, and wait examples
    with the following distribution:
    - 35% solution examples
    - 35% programming examples
    - 15% completion examples
    - 15% wait examples
    """
    
    # Load the base dataset
    data = load_dataset(dataset_name, split=split)
    
    # Define the distribution
    distribution = {
        'solution': 0.35,
        'programming': 0.35,
        'completion': 0.15,
        'wait': 0.15
    }
    
    # Use the prepare_combined_data function with programming system prompt
    return prepare_combined_data(
        data, 
        FULLSOLUTION_SYSTEM_PROMPT, 
        COMPLETION_SYSTEM_PROMPT, 
        PROGRAMMER_SYSTEM_PROMPT,
        tokenizer, 
        distribution)

# Get the formatted dataset with all types of examples
formatted_dataset = get_questions()
# Shuffle the combined dataset
formatted_dataset = formatted_dataset.shuffle(seed=20)
# Use a reasonable number of examples - smaller for notebook
formatted_dataset = formatted_dataset.select(range(500))
print(f"Dataset loaded with {len(formatted_dataset)} examples")

## Analyze Dataset

Let's analyze the dataset to understand its composition.

In [ ]:
# Count example types
example_types = {}
for example in formatted_dataset:
    et = example.get('example_type', 'unknown')
    example_types[et] = example_types.get(et, 0) + 1

# Create a bar chart of example types
plt.figure(figsize=(10, 6))
plt.bar(example_types.keys(), example_types.values(), color='skyblue')
plt.title('Distribution of Example Types')
plt.xlabel('Example Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

# Calculate percentages
total = sum(example_types.values())
percentages = {k: (v/total)*100 for k, v in example_types.items()}

# Display as a table
df = pd.DataFrame({
    'Count': example_types,
    'Percentage': {k: f"{v:.1f}%" for k, v in percentages.items()}
})
display(df)

## Examine Dataset Examples

Let's look at examples of each type in our dataset.

In [ ]:
# Function to find an example of a specific type
def find_example_by_type(dataset, example_type):
    for i, example in enumerate(dataset):
        if example.get('example_type') == example_type:
            return i, example
    return None, None

# Find examples of each type
example_types_list = ['solution', 'completion', 'programming', 'wait']
for et in example_types_list:
    idx, example = find_example_by_type(formatted_dataset, et)
    if example:
        print(f"\n{'-'*50}\nExample of type '{et}' (index {idx}):\n{'-'*50}")
        print(f"Problem: {example.get('problem', 'N/A')}")
        print(f"Answer: {example.get('answer', 'N/A')}")
        
        # Print prompt (truncated for readability)
        prompt = example.get('prompt', '')
        if len(prompt) > 200:
            prompt = prompt[:200] + "..."
        print(f"Prompt (truncated): {prompt}")
        
        # For completion examples, show partial solution
        if et == 'completion' and 'partial_solution' in example:
            partial = example.get('partial_solution', '')
            if len(partial) > 200:
                partial = partial[:200] + "..."
            print(f"Partial solution (truncated): {partial}")
            
        # Count tokens
        prompt_tokens = count_tokens(example.get('prompt', ''))
        print(f"Estimated prompt tokens: {prompt_tokens}")

## Analyze Dataset

Let's analyze the dataset to understand its composition.

In [ ]:
# Verify first few entries
solution_count = 0
completion_count = 0
wait_count = 0
programming_count = 0

for i in range(min(12, len(formatted_dataset))):
    entry = formatted_dataset[i]
    example_type = entry.get('example_type', 'unknown')
    
    if example_type == 'solution':
        solution_count += 1
    elif example_type == 'completion':
        completion_count += 1
    elif example_type == 'wait':
        wait_count += 1
    elif example_type == 'programming':
        programming_count += 1
        
    print(f"\nEntry {i} verification:")
    print(f"Type: {example_type}")
    print(f"Answer: {entry.get('answer')}")
    
    # Get token count for the prompt
    prompt = entry.get('prompt', '')
    prompt_tokens = count_tokens(prompt)
    print(f"Prompt tokens: {prompt_tokens}")
    
    if example_type == 'completion' and entry.get('partial_solution'):
        partial = entry.get('partial_solution')
        # Count steps in partial solution
        step_count = len(re.findall(r'<step>', partial))
        print(f"Steps in partial solution: {step_count}")
        
    elif example_type == 'wait':
        # Extract thinking section to verify wait modification
        thinking_pattern = re.compile(r'<thinking>(.*?)</thinking>', re.DOTALL)
        thinking_match = thinking_pattern.search(prompt)
    
    # Check for prompt indicators
    has_continue = 'continue' in prompt.lower()
    has_next_step = 'next step' in prompt.lower()
    has_wait = 'wait a second' in prompt.lower()
    print(f"Prompt indicators: continue={has_continue}, next_step={has_next_step}, wait={has_wait}")

print(f"\nSample ratio: {solution_count} solution examples, {completion_count} completion examples, {wait_count} wait examples, {programming_count} programming examples")

# Create a bar chart of example types
example_types = {'solution': solution_count, 'completion': completion_count, 'wait': wait_count, 'programming': programming_count}
plt.figure(figsize=(10, 6))
plt.bar(example_types.keys(), example_types.values(), color='skyblue')
plt.title('Distribution of Example Types in Sample')
plt.xlabel('Example Type')
plt.ylabel('Count')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Configure Training

Now let's set up the GRPO training configuration.

In [ ]:
# Log the dataset structure before training
logger.info("Dataset structure before training:")
sample_example = formatted_dataset[0]
for key, value in sample_example.items():
    logger.info(f"  {key}: {type(value)} - {value}")

# Count example types in the dataset
example_types = {}
for example in formatted_dataset:
    et = example.get('example_type', 'unknown')
    example_types[et] = example_types.get(et, 0) + 1

logger.info(f"Example types in dataset: {example_types}")

# GRPO specific training arguments - using smaller values for the notebook
training_args = GRPOConfig(
    torch_empty_cache_steps=1,
    learning_rate=6e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    optim="adamw_torch",
    logging_steps=1,
    bf16=is_bfloat16_supported(),
    fp16=not is_bfloat16_supported(),
    per_device_train_batch_size=2,  # Smaller for notebook
    gradient_accumulation_steps=2,  # Smaller for notebook
    num_generations=3,              # Smaller for notebook
    max_prompt_length=2048,
    max_completion_length=2548,
    num_train_epochs=1,
    max_steps=20,                   # Limit steps for notebook
    save_steps=10,
    max_grad_norm=0.1,
    report_to=report_to if use_wandb else "none",
    output_dir=output_dir,
)

print(f"Training configuration set up with {training_args.max_steps} steps")

## Initialize Trainer

Now let's initialize the GRPO trainer with our model, dataset, and reward function.

In [ ]:
# Initialize our custom callback
notebook_callback = LoggingCallback(reward_func=reward_func, logger=logger, save_frequency=1)

# Initialize trainer with reward function
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[reward_func],
    args=training_args,
    train_dataset=formatted_dataset,
    callbacks=[notebook_callback]
)

print("Trainer initialized and ready for training")

## Start Training

Now let's start the training process.

In [ ]:
# Train
try:
    trainer.train()
    logger.info("Training completed successfully")
except Exception as e:
    logger.error(f"Training failed: {str(e)}")
    if use_wandb:
        wandb.finish()
    raise

## Visualize Training Results

Let's visualize the training metrics.

In [ ]:
# Plot the metrics
notebook_callback.plot_metrics()

# Print final reward statistics
if hasattr(reward_func, 'stats'):
    print("\nFinal Reward Statistics Summary:")
    print(reward_func.stats.get_summary())
    
    print("\nReward Components:")
    for key, value in reward_func.stats.reward_components.items():
        print(f"  {key}: {value}")
    
    # Check for other stat categories
    for category in ['group_stats', 'step_stats', 'similarity_stats', 'programming_stats']:
        if hasattr(reward_func.stats, category):
            stats_dict = getattr(reward_func.stats, category)
            if stats_dict:
                print(f"\n{category.replace('_', ' ').title()}:")
                for key, value in stats_dict.items():
                    print(f"  {key}: {value}")
else:
    print("No statistics available.")

## Save Model

Finally, let's save the trained model.

In [ ]:
# Save model
try:
    models_dir = "models"
    os.makedirs(os.path.join(models_dir, reward_config.model_type), exist_ok=True)
    model_output_dir = os.path.join(models_dir, reward_config.model_type, timestamp)
    model.save_pretrained_merged(model_output_dir, tokenizer, save_method="merged_16bit")
    logger.info(f"Merged model saved to {model_output_dir}")
    print(f"Model saved to {model_output_dir}")
except Exception as e:
    logger.error(f"Failed to save model: {str(e)}")
    print(f"Error saving model: {str(e)}")
finally:
    if use_wandb:
        wandb.finish()
        print("Wandb logging finished")

## Conclusion

This notebook has demonstrated the complete training process for a Qwen model using GRPO with dynamic rewards. We've seen how to:

1. Configure the reward function
2. Prepare a dataset with different example types
3. Initialize and configure the model with LoRA
4. Set up the GRPO training process
5. Train the model and visualize the results
6. Save the trained model

This interactive approach allows for better monitoring and understanding of the training process compared to running the script directly.